# Make All Tables for Main Manuscript

Kendra Wyant  
March 5, 2026

In [ ]:

suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(source("https://github.com/jjcurtin/lab_support/blob/main/format_path.R?raw=true"))
library(kableExtra)



Attaching package: 'kableExtra'

The following object is masked from 'package:dplyr':

    group_rows


Attaching package: 'janitor'

The following objects are masked from 'package:stats':

    chisq.test, fisher.test

In [ ]:
study_dates <- read_csv(file.path(path_shared, "study_dates_gps.csv"), 
                        show_col_types = FALSE)

location <- read_csv(file.path(path_shared, "features_fairness.csv"), 
                        show_col_types = FALSE)

intake <- read_csv(file.path(path_shared, "survey_intake.csv"), 
                   show_col_types = FALSE) |>
  filter(subid %in% study_dates$subid) 


# Calcs to make df for table 1 (demographics and clinical characteristics)
n_total <- nrow(study_dates)

dem_age <-  intake |>
  select(var = age) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |> 
  add_row(var = "Age", .before = 1)

dem_gender <-  intake |>
  select(var = gender) |>
  filter(var != "Prefer not to say") |> 
  mutate(var = factor(var, 
                      levels = c("Man",
                                 "Woman",
                                 "Non-binary",
                                 "Not listed above"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |> 
  add_row(var = "Gender", .before = 1)

dem_orientation <-  intake |>
  select(var = orientation) |>
  filter(!is.na(var)) |> 
  mutate(var = factor(var, levels = c("Straight, that is, not gay or lesbian",
                                      "Lesbian or gay",
                                      "Bisexual",
                                      "Not sure",
                                      "Not listed above"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |> 
  add_row(var = "Orientation", .before = 1)

dem_race <- intake |>
  select(starts_with("race_"), -race_other_text) |> 
  pivot_longer(1:7, names_to = "var") |> 
  mutate(var = case_match(var,
                           "race_ai_an" ~ "American Indian/Alaskan Native",
                           "race_asian" ~ "Asian",
                           "race_black" ~ "Black/African American",
                           "race_nat_hi" ~ "Native Hawaiian/Other Pacific Islander",
                           "race_white" ~ "White/Caucasian",
                           "race_hispanic" ~ "Hispanic, Latino, or Spanish origin",
                           "race_other" ~ "Not listed above"),
         var = factor(var,levels = c("American Indian/Alaskan Native",
                                     "Asian",
                                     "Black/African American",
                                     "Native Hawaiian/Other Pacific Islander",
                                     "White/Caucasian",
                                     "Hispanic, Latino, or Spanish origin",
                                     "Not listed above"))) |> 
  group_by(var) |>
  count(value) |> 
  ungroup() |> 
  filter(value == "yes") |> 
  select(-value) |> 
  mutate(perc = (n / n_total) * 100) |>
  add_row(var = "Race and Ethnicity (select all that apply)", .before = 1)

dem_geography <- location |> 
  mutate(geography = if_else(geography == "urban/suburban", "urban", "rural")) |> 
  select(var = geography) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |> 
  add_row(var = "Geographic Location", .before = 1)
  

dem_education <- intake |>
  select(var = education) |>
  filter(!is.na(var)) |> 
  mutate(var = factor(var, 
                      levels = c("8th grade or less",
                                 "Some high school, but did not graduate",
                                 "High school graduate or GED",
                                 "Some college or 2-year degree",
                                 "4-year college graduate",
                                 "More than 4-year or advanced degree"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  add_row(var = "Education", .before = 1)

dem_employment <- intake |>
  select(var = employment) |>
  filter(!is.na(var)) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  add_row(var = "Employment", .before = 1)

dem_income <- intake |>
  select(var = income) |>
  filter(!is.na(var)) |> 
  mutate(var = factor(var, 
                      levels = c("Less than $25,000",
                                 "$25,000 - $34, 999",
                                 "$35,000 - $49,999",
                                 "$50,000 - $74, 999",
                                 "$75, 000 - $99, 999",
                                 "$100,000 - $149,999",
                                 "$150, 000 - $199,999",
                                 "$200, 000 or more"),
                      labels = c("Less than $25,000",
                                 "$25,000 - $34,999",
                                 "$35,000 - $49,999",
                                 "$50,000 - $74,999",
                                 "$75,000 - $99,999",
                                 "$100,000 - $149,999",
                                 "$150,000 - $199,999",
                                 "$200,000 or more"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  add_row(var = "Household Income", .before = 1)


dsm <- intake |>
  summarise(mean = mean(dsm_c, na.rm = TRUE),
            SD = sd(dsm_c, na.rm = TRUE),
            min = min(dsm_c, na.rm = TRUE),
            max = max(dsm_c, na.rm = TRUE)) |>
  mutate(var = "DSM-5 OUD Symptom Count",
        n = as.numeric(""),
        perc = as.numeric("")) |>
  select(var, n, perc, everything())

past_month_opioid <- intake |>
  select(var = whoassist_month_1_opioid) |>
  mutate(var = factor(var, levels = c("no", "yes"),
                      labels = c("No", "Yes"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  drop_na() |> 
  add_row(var = "Past Month Opioid Use", .before = 1)

past_month_detox <- intake |>
  select(var = other_tx_2) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  drop_na() |> 
  add_row(var = "Past Month Detox or Residential Treatment", .before = 1)

past_month_psych <- intake |>
  select(var = other_tx_1) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  drop_na() |> 
  add_row(var = "Past Month Psychiatric Medication", .before = 1)

opioid_preferred <- intake |>
  select(var = whoassist_opioid_3) |>
  mutate(var = factor(var,
                      levels = c("Fentanyl (Sublimaze, Actiq, apache, goodfella, TNT)",
                                 "Heroin (black tar, black pearl, black, china white, dope, white lady, smack, snow, speedball)",
                                 "Prescription opioid not for opioid treatment (OxyContin, Vicodin, Percocet, Opana)",
                                 "Medication for opioid treatment (e.g., Suboxone, Methadone, etc.)"),
                      labels = c("Fentanyl",
                                 "Heroin",
                                 "Prescription opioid not for opioid treatment",
                                 "Medication for opioid treatment"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  drop_na() |> 
  add_row(var = "Preferred Opioid", .before = 1)

roa_preferred <- intake |>
  select(var = whoassist_opioid_4) |>
  mutate(var = factor(var, 
                      levels = c("Injection (IV, muscle, under skin)",
                                 "Oral (swallow, chew)",
                                 "Smoke (chase)",
                                 "Sniff or snort",
                                 "Other"),
                      labels = c("Injection",
                                 "Oral",
                                 "Smoke",
                                 "Sniff or snort",
                                 "Other"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  drop_na() |> 
  add_row(var = "Preferred Route of Administration", .before = 1)

od <- intake |>
  select(var = whoassist_life_overdose) |>
  mutate(var = factor(var, 
                      levels = c("No",
                                 "1 time",
                                 "2-3 times",
                                 "4-5 times",
                                 "More than 5 times"),
                      labels = c("Never",
                                 "1 time",
                                 "2-3 times",
                                 "4-5 times",
                                 "More than 5 times"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  drop_na() |> 
  add_row(var = "Lifetime History of Overdose", .before = 1)

dsm <- intake |>
  select(var = dsm_c) |>
  mutate(var = case_match(var,
                          0 ~ "Under threshold (0-1)",
                          1 ~ "Under threshold (0-1)",
                          2 ~ "Mild (2-3)",
                          3 ~ "Mild (2-3)",
                          4 ~ "Moderate (4-5)",
                          5 ~ "Moderate (4-5)",
                          6 ~ "Severe (6+)",
                          7 ~ "Severe (6+)",
                          8 ~ "Severe (6+)",
                          9 ~ "Severe (6+)",
                          10 ~ "Severe (6+)",
                          11 ~"Severe (6+)"),
         var = factor(var, levels = c("Under threshold (0-1)",
                                      "Mild (2-3)",
                                      "Moderate (4-5)",
                                      "Severe (6+)"))) |> 
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100) |>
  drop_na() |> 
  add_row(var = "Self-reported DSM-5 OUD Symptom Count", .before = 1) |> 
  add_row(tibble(var = "Mild (2-3)", n = 0, perc = 0), .before = 3)


table_dem <- dem_age |> 
  bind_rows(dem_gender) |> 
  bind_rows(dem_orientation) |> 
  bind_rows(dem_race) |> 
  bind_rows(dem_geography) |> 
  bind_rows(dem_education) |> 
  bind_rows(dem_employment) |> 
  bind_rows(dem_income) |> 
  bind_rows(dsm) |>
  bind_rows(past_month_opioid) |> 
  bind_rows(past_month_detox) |> 
  bind_rows(past_month_psych) |> 
  bind_rows(opioid_preferred) |> 
  bind_rows(roa_preferred) |> 
  bind_rows(od) |> 
  mutate(perc = round(perc, 1)) |> 
  rename(` ` = var,
         N = n,
         `%` = perc)


In [ ]:
path_models <- format_path(str_c("risk2/models/combined"))

table_ci <- read_csv(here::here(path_models, "contrast_ablate_v10.csv"),
                    show_col_types = FALSE) 


### Table 1: Demographics and Clinical Characteristics

In [ ]:

footnote_table_dem_a <- "N = 296"


table_dem |> 
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::group_rows(start_row = 2, end_row = 7) |> 
  kableExtra::group_rows(start_row = 9, end_row = 12) |> 
  kableExtra::group_rows(start_row = 14, end_row = 18) |> 
  kableExtra::group_rows(start_row = 20, end_row = 26) |> 
  kableExtra::group_rows(start_row = 28, end_row = 29) |> 
  kableExtra::group_rows(start_row = 31, end_row = 36) |> 
  kableExtra::group_rows(start_row = 38, end_row = 43) |> 
  kableExtra::group_rows(start_row = 45, end_row = 52) |> 
  kableExtra::group_rows(start_row = 54, end_row = 57) |> 
  kableExtra::group_rows(start_row = 59, end_row = 60) |> 
  kableExtra::group_rows(start_row = 62, end_row = 63) |> 
  kableExtra::group_rows(start_row = 65, end_row = 66) |> 
  kableExtra::group_rows(start_row = 68, end_row = 71) |>
  kableExtra::group_rows(start_row = 73, end_row = 77) |>
  kableExtra::group_rows(start_row = 79, end_row = 83) |>
  kableExtra::footnote(general = c(footnote_table_dem_a), escape=FALSE)


Warning in attr(x, "align"): 'xfun::attr()' is deprecated.
Use 'xfun::attr2()' instead.
See help("Deprecated")
Warning in attr(x, "align"): 'xfun::attr()' is deprecated.
Use 'xfun::attr2()' instead.
See help("Deprecated")

N 
 % 
 
 
 
 
 Age 
 
 
 
 
 
 22-25 
 7 
 2.4 
 
 
 26-35 
 105 
 35.5 
 
 
 36-45 
 107 
 36.1 
 
 
 46-55 
 60 
 20.3 
 
 
 56-65 
 13 
 4.4 
 
 
 Over 65 
 4 
 1.4 
 
 
 Gender 
 
 
 
 
 
 Man 
 158 
 53.4 
 
 
 Woman 
 131 
 44.3 
 
 
 Non-binary 
 4 
 1.4 
 
 
 Not listed above 
 2 
 0.7 
 
 
 Orientation 
 
 
 
 
 
 Straight, that is, not gay or lesbian 
 237 
 80.1 
 
 
 Lesbian or gay 
 17 
 5.7 
 
 
 Bisexual 
 35 
 11.8 
 
 
 Not sure 
 1 
 0.3 
 
 
 Not listed above 
 4 
 1.4 
 
 
 Race and Ethnicity (select all that apply) 
 
 
 
 
 
 American Indian/Alaskan Native 
 17 
 5.7 
 
 
 Asian 
 3 
 1.0 
 
 
 Black/African American 
 43 
 14.5 
 
 
 Native Hawaiian/Other Pacific Islander 
 3 
 1.0 
 
 
 White/Caucasian 
 235 
 79.4 
 
 
 Hispanic, Latino, or Spanish origin 
 26 
 8.8 
 
 
 Not listed above 
 5 
 1.7 
 
 
 Geographic Location 
 
 
 
 
 
 rural 
 62 
 20.9 
 
 
 urban 
 274 
 92.6 
 
 
 Education 
 
 
 
 
 
 8th grade or less 
 5 
 1.7 
 
 
 Some high school, but did not graduate 
 25 
 8.4 
 
 
 High school graduate or GED 
 92 
 31.1 
 
 
 Some college or 2-year degree 
 130 
 43.9 
 
 
 4-year college graduate 
 30 
 10.1 
 
 
 More than 4-year or advanced degree 
 12 
 4.1 
 
 
 Employment 
 
 
 
 
 
 Disabled, not able to work 
 34 
 11.5 
 
 
 Employed, working 1-39 hours per week 
 69 
 23.3 
 
 
 Employed, working 40 or more hours per week 
 54 
 18.2 
 
 
 Not employed, NOT looking for work 
 32 
 10.8 
 
 
 Not employed, looking for work 
 102 
 34.5 
 
 
 Retired 
 3 
 1.0 
 
 
 Household Income 
 
 
 
 
 
 Less than $25,000 
 164 
 55.4 
 
 
 $25,000 - $34,999 
 42 
 14.2 
 
 
 $35,000 - $49,999 
 38 
 12.8 
 
 
 $50,000 - $74,999 
 29 
 9.8 
 
 
 $75,000 - $99,999 
 13 
 4.4 
 
 
 $100,000 - $149,999 
 3 
 1.0 
 
 
 $150,000 - $199,999 
 4 
 1.4 
 
 
 $200,000 or more 
 1 
 0.3 
 
 
 Self-reported DSM-5 OUD Symptom Count 
 
 
 
 
 
 Under threshold (0-1) 
 5 
 1.7 
 
 
 Mild (2-3) 
 0 
 0.0 
 
 
 Moderate (4-5) 
 5 
 1.7 
 
 
 Severe (6+) 
 284 
 95.9 
 
 
 Past Month Opioid Use 
 
 
 
 
 
 No 
 201 
 67.9 
 
 
 Yes 
 93 
 31.4 
 
 
 Past Month Detox or Residential Treatment 
 
 
 
 
 
 No 
 189 
 63.9 
 
 
 Yes 
 106 
 35.8 
 
 
 Past Month Psychiatric Medication 
 
 
 
 
 
 No 
 151 
 51.0 
 
 
 Yes 
 144 
 48.6 
 
 
 Preferred Opioid 
 
 
 
 
 
 Fentanyl 
 50 
 16.9 
 
 
 Heroin 
 122 
 41.2 
 
 
 Prescription opioid not for opioid treatment 
 93 
 31.4 
 
 
 Medication for opioid treatment 
 28 
 9.5 
 
 
 Preferred Route of Administration 
 
 
 
 
 
 Injection 
 104 
 35.1 
 
 
 Oral 
 56 
 18.9 
 
 
 Smoke 
 48 
 16.2 
 
 
 Sniff or snort 
 83 
 28.0 
 
 
 Other 
 3 
 1.0 
 
 
 Lifetime History of Overdose 
 
 
 
 
 
 Never 
 119 
 40.2 
 
 
 1 time 
 41 
 13.9 
 
 
 2-3 times 
 69 
 23.3 
 
 
 4-5 times 
 26 
 8.8 
 
 
 More than 5 times 
 39 
 13.2 
 
 
 
 Note: 
 
 N = 296

### Table 2: Model Comparisons

In [ ]:

footnote_table_model <- "Median auROC differences greater than 0 indicate the full model, on average, performed better than the ablated model (i.e., full vs. ablated daily survey, full vs. ablated geolocation, full vs. ablated daily survey and geolocation). Bayesian CI represents the range of values where there is a 95% probability that the true auROC difference lies within that range. Probability indicates the posterior probability that this difference is greater than 0 (i.e., the full model is performing better)."

table_ci |> 
  mutate(`Bayesian CI` = str_c("[", round(lower, 3), ", ", round(upper, 3), "]"),
         Median = as.character(round(median, 3)),
         Probability = sprintf("%.3f", probability)) |> 
  select(Contrast = contrast, Median, `Bayesian CI`, Probability) |> 
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::column_spec(1, width = "25em") |> 
  kableExtra::add_footnote(label = footnote_table_model,
                           notation = "none",
                           escape = FALSE)


Warning in attr(x, "align"): 'xfun::attr()' is deprecated.
Use 'xfun::attr2()' instead.
See help("Deprecated")
Warning in attr(x, "align"): 'xfun::attr()' is deprecated.
Use 'xfun::attr2()' instead.
See help("Deprecated")

Contrast 
 Median 
 Bayesian CI 
 Probability 
 
 
 
 
 full vs. ablated daily survey 
 0.294 
 [0.271, 0.317] 
 1.000 
 
 
 full vs. ablated geolocation 
 -0.002 
 [-0.012, 0.007] 
 0.349 
 
 
 full vs. ablated daily survey and geolocation 
 0.363 
 [0.339, 0.388] 
 1.000 
 
 
 
 
 Median auROC differences greater than 0 indicate the full model, on average, performed better than the ablated model (i.e., full vs. ablated daily survey, full vs. ablated geolocation, full vs. ablated daily survey and geolocation). Bayesian CI represents the range of values where there is a 95% probability that the true auROC difference lies within that range. Probability indicates the posterior probability that this difference is greater than 0 (i.e., the full model is performing better).